In [1]:
import os
from pathlib import Path

In [2]:
path = Path("../data/responses_1/")
all_ids = [os.path.basename(p).replace(".json", "") for p in  path.rglob('*.json')]
all_ids = set(all_ids)
print(len(all_ids))

KeyboardInterrupt: 

In [3]:
import json
citation_index = json.load(open("../data/output/citation_index.json"))

In [ ]:
print(f"Number of unique nodes in the citation network: {len(set(citation_index.keys()))}")

total_edges = 0
edges_with_downloaded_ids = 0

for node, info in citation_index.items():
    index = info.get("index", [])
    reverse_index = info.get("reverse_index", [])
    if index:
        total_edges += len(index)
        edges_with_downloaded_ids += sum(1 for cited in index if cited in all_ids)
    del index
    del reverse_index

Number of unique nodes in the citation network: 8953263


In [9]:
print(f"Number of edges in the citation network: {total_edges}")
print(f"Number of edges containing ids in the ID set: {edges_with_downloaded_ids}")

Number of edges in the citation network: 47991643
Number of edges containing ids in the ID set: 22123799


In [ ]:
print(f"Number of unique nodes in the citation network and in the ID set: {len(set(citation_index.keys()) & institution_ids)}")

Number of unique nodes in the citation network and in the ID set: 1680563


In [ ]:
path = Path("../data/responses_1/")
institution_ids = [os.path.basename(p).replace(".json", "") for p in  path.rglob('*.json')]
institution_ids = set(institution_ids)
print(len(institution_ids))

In [ ]:
institution_paths = [
    Path("../data/responses_1/by_institution/universidade_federal_rio_janei"),
    Path("../data/responses_1/by_institution/massachusetts_institute_techno"),
    Path("../data/responses_1/by_institution/universidade_federal_sao_carlo"),
    Path("../data/responses_1/by_institution/carnegie_mellon_university"),
    Path("../data/responses_1/by_institution/universidade_sao_paulo"),
    Path("../data/responses_1/by_institution/nanjing_university"),
    Path("../data/responses_1/by_institution/university_california_berkeley"),
    Path("../data/responses_1/by_institution/eindhoven_university_technolog"),
    Path("../data/responses_1/by_institution/national_university_singapore"),
    Path("../data/responses_1/by_institution/federal_university_minas_gerai"),
    Path("../data/responses_1/by_institution/harvard_university"),
    Path("../data/responses_1/by_institution/peking_university"),
    Path("../data/responses_1/by_institution/university_melbourne"),
    Path("../data/responses_1/by_institution/humboldt-universitat_berlin"),
    Path("../data/responses_1/by_institution/university_oxford"),
    Path("../data/responses_1/by_institution/university_pennsylvania"),
    Path("../data/responses_1/by_institution/stanford_university"),
    Path("../data/responses_1/by_institution/university_tokyo"),
    Path("../data/responses_1/by_institution/instituto_tecnologico_aeronaut"),
    Path("../data/responses_1/by_institution/technical_university_darmstadt"),
    Path("../data/responses_1/by_institution/institut_polytechnique_paris"),
    Path("../data/responses_1/by_institution/technical_university_munich"),
    Path("../data/responses_1/by_institution/vellore_institute_technology_v"),
    Path("../data/responses_1/by_institution/wuhan_university"),
    Path("../data/responses_1/by_institution/king_s_college_london"),
    Path("../data/responses_1/by_institution/tsinghua_university"),
    Path("../data/responses_1/by_institution/yale_university"),
    Path("../data/responses_1/by_institution/kth_royal_institute_technology"),
    Path("../data/responses_1/by_institution/unesp"),
    Path("../data/responses_1/by_institution/kyoto_university")
]

In [ ]:
institution_ids = set()
for path in institution_paths:
    institution_ids.update([os.path.basename(p).replace(".json", "") for p in path.rglob('*.json')])

In [ ]:
print(f"Number of unique institution ids: {len(institution_ids)}")

Number of unique institution ids: 153464


In [ ]:
unicamp_path = Path("../data/responses_1/_unicamp_cs")
unicamp_ids = set(os.path.basename(p).replace(".json", "") for p in unicamp_path.rglob('*.json'))
print(f"Number of unique institution ids in unicamp: {len(unicamp_ids)}")

top_tier_path = Path("../data/responses_1/_top_cited_cs")
top_tier_ids = set(os.path.basename(p).replace(".json", "") for p in top_tier_path.rglob('*.json'))
print(f"Number of unique institution ids in top_tier: {len(top_tier_ids)}")

Number of unique institution ids in unicamp: 2018
Number of unique institution ids in top_tier: 461


In [ ]:
from dotenv import load_dotenv
load_dotenv("../.env")

True

In [ ]:
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE")

In [ ]:
from neo4j import GraphDatabase
neo4j_driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
all_articles_query = """
MATCH (a:Article)
RETURN a.openalex_id
"""
with neo4j_driver.session(database=NEO4J_DATABASE) as session:
    result = session.run(all_articles_query)
    article_ids_in_neo4j = set(record["a.openalex_id"] for record in result)
print(f"Number of unique article nodes in Neo4j: {len(article_ids_in_neo4j)}")

Number of unique article nodes in Neo4j: 49196


In [ ]:
all_citations_query = """
MATCH (a:Article)-[:CITES]->(b:Article)
RETURN a.openalex_id AS citing, b.openalex_id AS cited
"""
with neo4j_driver.session(database=NEO4J_DATABASE) as session:
    result = session.run(all_citations_query)
    citations_in_neo4j = set((record["citing"], record["cited"]) for record in result)
print(f"Number of unique citation edges in Neo4j: {len(citations_in_neo4j)}")

In [ ]:
print(f"Number of articles from institutions in the ID set that are present in Neo4j: {len(article_ids_in_neo4j & institution_ids)}")
print(f"Number of articles from Unicamp that are present in Neo4j: {len(article_ids_in_neo4j & unicamp_ids)}")
print(f"Number of articles from top tier that are present in Neo4j: {len(article_ids_in_neo4j & top_tier_ids)}")

Number of articles from institutions in the ID set that are present in Neo4j: 6207
Number of articles from Unicamp that are present in Neo4j: 1907
Number of articles from top tier that are present in Neo4j: 298


In [ ]:
removed_wids = all_ids.difference(article_ids_in_neo4j)
print(f"Number of article nodes in Neo4j that are not in any of the ID sets: {len(removed_wids)}")

Number of article nodes in Neo4j that are not in any of the ID sets: 1631369


In [ ]:
import json

def _strip(url: str) -> str:
    return url.replace("https://openalex.org/", "") if url else ""

def extract_institutions(raw: dict) -> list[dict]:
    seen, institutions = set(), []
    for a in raw.get("authorships", []):
        for inst in a.get("institutions", []):
            iid = inst.get("id", "")
            if not iid or iid in seen:
                continue
            seen.add(iid)
            institutions.append({
                "openalex_id":  _strip(iid),
                "display_name": inst.get("display_name"),
            })
    return institutions


def extract_venue(raw: dict) -> dict | None:
    loc    = raw.get("primary_location") or {}
    source = loc.get("source") or {}
    sid    = source.get("id", "")
    if not sid:
        return None
    return {
        "openalex_id":  _strip(sid),
        "display_name": source.get("display_name"),
    }


def extract_subfields(raw: dict) -> list[dict]:
    seen, subfields = set(), []
    for t in raw.get("topics", []):
        subfield = t.get("subfield") or {}
        sid = subfield.get("id", "")
        if not sid or sid in seen:
            continue
        seen.add(sid)
        subfields.append({
            "openalex_id":  _strip(sid),
            "display_name": subfield.get("display_name"),
        })
    return subfields

def get_info_removed(removed_wids):
    # Get count by institutions, subfields and venues from the articles that were removed.
    # Good to understand if there are any patterns in the removed articles.
    rm_institutions = {} # iid -> (display_name, count)
    rm_subfields = {} # iid -> (display_name, count)
    rm_venues = {} # iid -> (display_name, count)
    processed = []
    if os.path.exists("removed_wids_info.json"):
        with open("removed_wids_info.json", "r", encoding="utf-8") as f:
            data = json.load(f)
            processed = data.get("processed", [])
            rm_institutions = data.get("institutions", {})
            rm_subfields = data.get("subfields", {})
            rm_venues = data.get("venues", {})
        print(f"Loaded info for {len(rm_institutions)} institutions, {len(rm_subfields)} subfields and {len(rm_venues)} venues from file.")
    for wid in removed_wids:
        if wid in processed:
            continue
        processed.append(wid)
        if len(processed) % 100000 == 0:
            with open("removed_wids_info.json", "w", encoding="utf-8") as f:
                json.dump({
                    "processed": processed,
                    "institutions": rm_institutions,
                    "subfields": rm_subfields,
                    "venues": rm_venues
                }, f, ensure_ascii=False, indent=2)
            print(f"Processed {len(processed)} / {len(removed_wids)} removed articles...")
            # return rm_institutions, subfields, rm_venues
        path = "../data/responses_1/" + wid[:3] + "/" + wid[3:5] + "/" + wid + ".json"
        if not os.path.exists(path):
            # print(f"File not found for wid {wid}: {path}")
            continue
        with open(path, "r", encoding="utf-8") as f:
            info = json.load(f)
            subfields = extract_subfields(info)
            venue = extract_venue(info)
            insts = extract_institutions(info)
            for inst in insts:
                iid = inst["openalex_id"]
                rm_institutions[iid] = (inst["display_name"], rm_institutions.get(iid, (None, 0))[1] + 1)
            for sub in subfields:
                sid = sub["openalex_id"]
                rm_subfields[sid] = (sub["display_name"], rm_subfields.get(sid, (None, 0))[1] + 1)
            if venue:
                vid = venue["openalex_id"]
                rm_venues[vid] = (venue["display_name"], rm_venues.get(vid, (None, 0))[1] + 1)

    return rm_institutions, rm_subfields, rm_venues

In [ ]:
rm_institutions, subfields, rm_venues = get_info_removed(removed_wids)

Loaded info for 15112 institutions, 232 subfields and 10628 venues from file.


In [ ]:
for inst, count in sorted(rm_institutions.values(), key=lambda x: x[1], reverse=True):
    print(f"{inst}: {count}")   

Centre National de la Recherche Scientifique: 1055
Carnegie Mellon University: 1039
Chinese Academy of Sciences: 894
Tsinghua University: 872
Stanford University: 832
University of California, Berkeley: 805
Massachusetts Institute of Technology: 780
University of Illinois Urbana-Champaign: 769
Nanyang Technological University: 596
National University of Singapore: 581
Georgia Institute of Technology: 575
Microsoft (United States): 562
University of Maryland, College Park: 548
Shanghai Jiao Tong University: 526
IBM (United States): 526
University of Washington: 511
Institut national de recherche en informatique et en automatique: 506
University of Waterloo: 492
University of Toronto: 482
Peking University: 473
The University of Texas at Austin: 472
University of Southern California: 458
University of Michigan: 458
Princeton University: 450
Purdue University West Lafayette: 445
Zhejiang University: 437
Beihang University: 416
University of Science and Technology of China: 413
University 

In [ ]:
for sub, count in sorted(subfields.values(), key=lambda x: x[1], reverse=True):
    print(f"{sub}: {count}")

AttributeError: 'list' object has no attribute 'values'

In [ ]:
for ven, count in sorted(rm_venues.values(), key=lambda x: x[1], reverse=True):
    print(f"{ven}: {count}")

IEEE Access: 1018
Neurocomputing: 499
Pattern Recognition: 441
Expert Systems with Applications: 439
Physical Review A: 422
Information Sciences: 397
Sensors: 394
IEEE Transactions on Image Processing: 371
IEEE Transactions on Information Theory: 364
Physical Review Letters: 350
IEEE Transactions on Pattern Analysis and Machine Intelligence: 341
Proceedings of SPIE, the International Society for Optical Engineering/Proceedings of SPIE: 334
Theoretical Computer Science: 327
Proceedings of the AAAI Conference on Artificial Intelligence: 320
Multimedia Tools and Applications: 310
IEEE Transactions on Signal Processing: 300
arXiv (Cornell University): 289
IEEE Transactions on Computers: 279
IEEE Transactions on Circuits and Systems for Video Technology: 272
Knowledge-Based Systems: 256
Communications of the ACM: 252
Procedia Computer Science: 239
IEEE Transactions on Automatic Control: 233
Pattern Recognition Letters: 222
IEEE Internet of Things Journal: 219
Future Generation Computer Syst